# Step 8: HMM and Viterbi

Use the tuned ranker's per-point OOF probabilities to tune decoder settings. Reuse the same helpers for stacking once its held-out predictions arrive.

For speed, select **one complete training trip per saved fold**, closest to 700 GPS points. Selection uses trip sizes and folds only. This is a compact decoder-development sample, not an exhaustive search or an unbiased new CV estimate. No student-test label is used to choose settings.

Candidate misses remain wrong. Missing OOF emissions, including flagged groups, split the sequence; they are never filled using reference routes.

In [1]:
%run 08_hmm_helpers.ipynb
from datetime import datetime
from zoneinfo import ZoneInfo

model_folder = Path("results/step7_xgb/tuned_sample10000")
output = Path("results/hmm/ranker")
output.mkdir(parents=True, exist_ok=True)
contract = json.loads(Path("outputs/folds/contract.json").read_text())
summary = pd.read_csv("outputs/candidates/summary.csv", dtype={"trip_id": str})
folds = pd.read_csv("outputs/folds/trip_folds.csv", dtype={"trip_id": str})
summary = summary.merge(folds, on="trip_id", validate="one_to_one")
summary["size_difference"] = (summary["gps_points"] - 700).abs()
sample = summary.sort_values(["size_difference", "trip_id"]).groupby("fold").first().reset_index()
train_points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
train_points = train_points[train_points["trip_id"].isin(sample["trip_id"])].merge(
    pd.read_csv(contract["fold_file"], usecols=["group_id", "fold"]), on="group_id", validate="one_to_one")
ids = set(train_points["group_id"])
oof_parts = []
for chunk in pd.read_csv(model_folder / "oof_tuned_sample10000.csv", chunksize=250000):
    oof_parts.append(chunk[chunk["group_id"].isin(ids)])
oof = pd.concat(oof_parts, ignore_index=True)
assert (oof["fold"] == oof["group_id"].map(train_points.set_index("group_id")["fold"])).all()
sample.to_csv(output / "tuning_trips.csv", index=False)
print("Decoder tuning trips:", sample["trip_id"].tolist())

Decoder tuning trips: ['00000027', '00000037', '00000079', '00000045', '00000029']


## Tune on training OOF only

Try a 3×3 grid. Each trip is decoded independently, then correct selected segments are counted over its valid labelled points. Show per-fold metrics, then choose by pooled Top-1. Deterministic ties prefer smaller transition and emission weights.

In [2]:
if (output / "hmm_config.json").exists():
    config = json.loads((output / "hmm_config.json").read_text())
    assert config["tuning_source"] == str(model_folder / "oof_tuned_sample10000.csv")
    assert config["tuning_trips"] == sample["trip_id"].tolist()
    assert config["tuning_source_bytes"] == (model_folder / "oof_tuned_sample10000.csv").stat().st_size
    assert config["tuning_source_mtime_ns"] == (model_folder / "oof_tuned_sample10000.csv").stat().st_mtime_ns
    assert (output / "tuning_grid.csv").exists() and (output / "tuning_per_fold.csv").exists()
    print("Reusing frozen OOF-only decoder configuration:", config)
    del oof
else:
    prepared = {}
    for row in sample.itertuples(index=False):
        point_rows = train_points[train_points["trip_id"] == row.trip_id]
        scores = oof[oof["group_id"].isin(point_rows["group_id"])]
        network = load_network(row.trip_id)
        prepared[row.trip_id] = prepare_trip(scores, point_rows, network)
        print("Prepared training trip:", row.trip_id, flush=True)
    
    records = []
    for beta in [0.25, 1.0, 4.0]:
        for emission_weight in [0.5, 1.0, 2.0]:
            for row in sample.itertuples(index=False):
                decoded, _ = decode_trip(prepared[row.trip_id], beta, emission_weight, posterior=False)
                valid = train_points[(train_points["trip_id"] == row.trip_id) & ~train_points["bad_gps"]]
                joined = valid.merge(decoded[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
                correct = joined["true_arc_id"].eq(joined["segment_id"]).sum()
                records.append({"beta": beta, "emission_weight": emission_weight, "trip_id": row.trip_id,
                    "fold": row.fold, "points": len(valid), "correct": correct, "top1": correct / len(valid)})
            print("Tuned beta", beta, "emission weight", emission_weight, flush=True)
    trials = pd.DataFrame(records)
    trials.to_csv(output / "tuning_per_fold.csv", index=False)
    results = trials.groupby(["beta", "emission_weight"], as_index=False)[["points", "correct"]].sum()
    results["top1"] = results["correct"] / results["points"]
    results.to_csv(output / "tuning_grid.csv", index=False)
    best = results.sort_values(["top1", "beta", "emission_weight"], ascending=[False, True, True]).iloc[0]
    config = {"beta": float(best["beta"]), "emission_weight": float(best["emission_weight"]),
        "temperature": 1.0, "gap_reset_s": 30, "transition_distance_scale_floor_m": 20, "unreachable_penalty": 20,
        "transition_cost_cap": 100, "topology": "directed reachability through full graph",
        "transition_distance": "endpoint-bridge lower-bound surrogate; not shortest road distance",
        "tuning_source": str(model_folder / "oof_tuned_sample10000.csv"),
        "tuning_source_bytes": (model_folder / "oof_tuned_sample10000.csv").stat().st_size,
        "tuning_source_mtime_ns": (model_folder / "oof_tuned_sample10000.csv").stat().st_mtime_ns,
        "tuning_trips": sample["trip_id"].tolist(), "tuning_folds": sample["fold"].tolist(),
        "tuning_points": int(best["points"]), "tuning_top1": float(best["top1"]),
        "posterior": "forward-backward marginals; Viterbi joint path reported separately",
        "selected_at": datetime.now(ZoneInfo("Asia/Kolkata")).isoformat()}
    (output / "hmm_config.json").write_text(json.dumps(config, indent=2) + "\n")
    display(results.sort_values("top1", ascending=False))
    del prepared, oof


Prepared training trip: 00000027


Prepared training trip: 00000037


Prepared training trip: 00000079


Prepared training trip: 00000045


Prepared training trip: 00000029


Tuned beta 0.25 emission weight 0.5


... (5 outputs left out)

Tuned beta 4.0 emission weight 0.5


Tuned beta 4.0 emission weight 1.0


Tuned beta 4.0 emission weight 2.0


,beta,emission_weight,points,correct,top1
5,1.00,2.0,3433,3192,0.929799
2,0.25,2.0,3433,3183,0.927177
4,1.00,1.0,3433,3175,0.924847
1,0.25,1.0,3433,3167,0.922517
8,4.00,2.0,3433,3143,0.915526
3,1.00,0.5,3433,3075,0.895718
0,0.25,0.5,3433,3037,0.884649
7,4.00,1.0,3433,3001,0.874163
6,4.00,0.5,3433,2895,0.843286
